In [2]:
import cv2
import numpy as np
import pandas as pd
import time
from sklearn.tree import DecisionTreeClassifier
from IPython.display import display, Javascript, clear_output
from google.colab.output import eval_js
from base64 import b64decode
from google.colab.patches import cv2_imshow
import warnings
warnings.filterwarnings('ignore')

# ==========================================
# 1. ENTRENAMIENTO DE LA INTELIGENCIA ARTIFICIAL (Machine Learning)
# ==========================================
print("🧠 Entrenando al modelo de Machine Learning...")

# Generamos el dataset simulado (como en el ejemplo anterior)
hue_buenos = np.random.normal(loc=50, scale=8, size=500)
manchas_buenos = np.random.normal(loc=2, scale=1, size=500)
clase_buenos = np.ones(500) # 1 = Bueno

hue_malos = np.random.normal(loc=25, scale=8, size=500)
manchas_malos = np.random.normal(loc=15, scale=5, size=500)
clase_malos = np.zeros(500) # 0 = Malo

X_train = np.column_stack((np.concatenate([hue_buenos, hue_malos]),
                           np.concatenate([manchas_buenos, manchas_malos])))
y_train = np.concatenate([clase_buenos, clase_malos])

# Inicializamos y entrenamos el cerebro (Árbol de Decisión)
modelo_ia = DecisionTreeClassifier(max_depth=3)
modelo_ia.fit(X_train, y_train)

print("✅ IA entrenada y lista para clasificar limones.\n")

# ==========================================
# 2. INICIALIZAR CONTADORES GLOBALES
# ==========================================
if 'total_buenos' not in globals():
    total_buenos = 0
if 'total_malos' not in globals():
    total_malos = 0

# ==========================================
# 3. FUNCIÓN PARA FOTO AUTOMÁTICA (CADA 5 SEG)
# ==========================================
def take_photo_auto(filename='photo.jpg', quality=0.8):
    js = Javascript('''
        async function takePhotoAuto(quality) {
            const div = document.createElement('div');
            const info = document.createElement('h3');
            info.style.color = '#ff9800';
            info.style.fontFamily = 'sans-serif';
            info.textContent = '⏱️ Cámara activa: Tomando foto automáticamente en 5 segundos...';
            div.appendChild(info);

            const video = document.createElement('video');
            video.style.display = 'block';
            const stream = await navigator.mediaDevices.getUserMedia({video: true});

            document.body.appendChild(div);
            div.appendChild(video);
            video.srcObject = stream;
            await video.play();

            google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);

            return new Promise((resolve) => {
                setTimeout(() => {
                    const canvas = document.createElement('canvas');
                    canvas.width = video.videoWidth;
                    canvas.height = video.videoHeight;
                    canvas.getContext('2d').drawImage(video, 0, 0);
                    stream.getVideoTracks()[0].stop();
                    div.remove();
                    resolve(canvas.toDataURL('image/jpeg', quality));
                }, 5000);
            });
        }
        ''')
    display(js)
    data = eval_js('takePhotoAuto({})'.format(quality))
    binary = b64decode(data.split(',')[1])
    with open(filename, 'wb') as f:
        f.write(binary)
    return filename

# ==========================================
# 4. PROCESAMIENTO Y PREDICCIÓN CON ML
# ==========================================
def procesar_con_ia(imagen, modelo):
    hsv = cv2.cvtColor(imagen, cv2.COLOR_BGR2HSV)

    # Paso A: OpenCV solo busca "objetos" (aislar limones de la banda gris)
    # Ignoramos grises (baja saturación). Buscamos color (S > 40) o cosas muy oscuras (V < 60).
    mask_color = cv2.inRange(hsv, np.array([0, 40, 60]), np.array([180, 255, 255]))
    mask_black = cv2.inRange(hsv, np.array([0, 0, 0]), np.array([180, 255, 60]))
    mask_todos_limones = cv2.bitwise_or(mask_color, mask_black)

    buenos_en_foto = 0
    malos_en_foto = 0

    contornos, _ = cv2.findContours(mask_todos_limones, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    for c in contornos:
        if cv2.contourArea(c) > 500:
            # Paso B: Extraer las características de este limón en particular
            x, y, w, h = cv2.boundingRect(c)

            # Creamos una mini-máscara solo para el limón actual
            mascara_individual = np.zeros(hsv.shape[:2], dtype=np.uint8)
            cv2.drawContours(mascara_individual, [c], -1, 255, -1)

            # Feature 1: Tono Promedio (Hue)
            tono_promedio = cv2.mean(hsv[:,:,0], mask=mascara_individual)[0]

            # Feature 2: Porcentaje de Manchas (Píxeles oscuros)
            canal_brillo = hsv[:,:,2]
            pixeles_oscuros = cv2.countNonZero(cv2.bitwise_and(cv2.inRange(canal_brillo, 0, 80), mascara_individual))
            pixeles_totales = cv2.countNonZero(mascara_individual)
            porcentaje_manchas = (pixeles_oscuros / pixeles_totales) * 100 if pixeles_totales > 0 else 0

            # Paso C: La Inteligencia Artificial toma la decisión
            datos_del_limon = np.array([[tono_promedio, porcentaje_manchas]])
            prediccion_ia = modelo.predict(datos_del_limon)[0]

            # Paso D: Actuar según lo que diga la IA
            if prediccion_ia == 1.0: # La IA dice que es BUENO
                buenos_en_foto += 1
                color_caja = (0, 255, 0)
                etiqueta = f'BUENO (H:{int(tono_promedio)})'
            else: # La IA dice que es MALO
                malos_en_foto += 1
                color_caja = (0, 0, 255)
                etiqueta = f'MALO (H:{int(tono_promedio)})'

            cv2.rectangle(imagen, (x, y), (x+w, y+h), color_caja, 3)
            cv2.putText(imagen, etiqueta, (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color_caja, 2)

    # Ajustar tamaño para visualización
    alto, ancho = imagen.shape[:2]
    if ancho > 600:
        imagen = cv2.resize(imagen, (600, int(alto * (600/ancho))))

    cv2_imshow(imagen)
    return buenos_en_foto, malos_en_foto

# ==========================================
# 5. BUCLE INFINITO DE LA BANDA TRANSPORTADORA
# ==========================================
try:
    while True:
        clear_output(wait=True)
        print("====================================================")
        print("🤖 SISTEMA DE CLASIFICACIÓN CON MACHINE LEARNING")
        print("====================================================")
        print(f"📊 CONTEO ACUMULADO: {total_buenos} Buenos | {total_malos} Malos")
        print("====================================================")
        print("Presiona el botón de 'Stop' ⬛ en Colab para detener.\n")

        photo_path = take_photo_auto()
        img_camera = cv2.imread(photo_path)

        if img_camera is not None:
            # Le pasamos la imagen Y el modelo entrenado a la función
            b, m = procesar_con_ia(img_camera, modelo_ia)

            total_buenos += b
            total_malos += m

            print("\n--- DECISIÓN DE LA IA PARA ESTA FOTO ---")
            if b > 0: print(f"🟢 PIN VERDE: ON ({b} clasificados como Buenos)")
            if m > 0: print(f"🔴 PIN ROJO: ON ({m} clasificados como Malos)")
            if b == 0 and m == 0: print("⚪ PINES APAGADOS (Banda vacía)")

            print(f"\n✅ Total Acumulado - BUENOS: {total_buenos} | MALOS: {total_malos}")
            time.sleep(1)

        else:
            print("Error al leer la imagen capturada.")

except KeyboardInterrupt:
    print("\n========================================")
    print("🛑 CICLO DETENIDO POR EL USUARIO")
    print(f"TOTAL FINAL -> BUENOS: {total_buenos} | MALOS: {total_malos}")
    print("========================================")
except Exception as err:
    print(f"\nError: {str(err)}")

🤖 SISTEMA DE CLASIFICACIÓN CON MACHINE LEARNING
📊 CONTEO ACUMULADO: 1 Buenos | 20 Malos
Presiona el botón de 'Stop' ⬛ en Colab para detener.



<IPython.core.display.Javascript object>


🛑 CICLO DETENIDO POR EL USUARIO
TOTAL FINAL -> BUENOS: 1 | MALOS: 20
